# 1. Amazon EMR: The Big Picture

**Amazon EMR (Elastic MapReduce)** is a managed AWS service for running distributed data-processing frameworks on scalable infrastructure.

Use EMR when one machine is not enough, when processing must finish faster through parallelism, or when you want managed setup for engines such as Apache Spark, Hadoop, Hive, Trino, Flink, and HBase.

**Mental model:** data lives durably in Amazon S3; an EMR cluster supplies temporary compute; processing engines transform the data; results return to S3.

# 2. What EMR Manages

Without EMR, you provision machines, install compatible framework versions, configure networking, coordinate nodes, replace unhealthy machines, and collect logs yourself. EMR automates much of that operational work.

EMR provides:

- Release bundles containing tested big-data applications
- Cluster provisioning on Amazon EC2
- Bootstrap actions and application configuration
- Job submission through **steps**
- Monitoring through the EMR console, logs, and Amazon CloudWatch
- Scaling through instance groups, instance fleets, managed scaling, or custom policies

EMR manages the platform; you still design data layout, permissions, capacity, reliability, and cost controls.

# 3. Amazon S3: Meaning and Purpose

**S3 expands to Amazon Simple Storage Service.** It is durable, scalable **object storage**. Data is stored as objects inside buckets and addressed by keys, for example `s3://company-data/sales/year=2026/month=08/`.

Common uses with EMR:

- Store raw, cleaned, curated, and archived datasets
- Read inputs and write final outputs from Spark, Hive, and other engines
- Keep scripts, configuration files, and bootstrap resources
- Store application logs after the cluster disappears
- Build a data lake shared by many clusters and AWS analytics services

S3 separates storage from compute: the same durable dataset can be processed by a small cluster today and a larger cluster tomorrow.

# 4. S3 and HDFS: Key Differences

| Area | Amazon S3 | HDFS on EMR |
|---|---|---|
| Storage model | Object storage using buckets and keys | Distributed file system using blocks |
| Lifetime | Independent of the cluster | Tied to cluster instances |
| Durability approach | AWS-managed regional durability | Replication across cluster nodes |
| Compute coupling | Decoupled; many clusters can reuse data | Coupled to the current cluster |
| Best fit | Durable input, output, lake, archive | Fast scratch space, cache, intermediate data |
| Access behavior | Network/object operations; rename is not a native atomic filesystem rename | Locality-aware filesystem operations |
| Cost pattern | Pay for stored objects and requests/data transfer as applicable | Pay for EC2-attached storage while instances run |

EMR applications access S3 through connectors such as **S3A** and, depending on release/application configuration, EMRFS. Do not treat S3 as though it has every POSIX or HDFS filesystem behavior.

# 5. Recommended Storage Pattern

A robust processing flow is:

`S3 input → EMR compute → local/HDFS intermediate work → S3 output`

Keep authoritative data in S3. Use HDFS and local disks for temporary shuffle files, spills, caches, and intermediate results. Before termination, write every result worth keeping to S3 or another durable external system.

Example Spark paths:

```text
Read:  s3://analytics-raw/events/
Write: s3://analytics-curated/daily_events/
```

Use IAM roles and bucket policies to grant only the required access. Enable encryption and organize keys with partition-friendly prefixes.

# 6. EMR Architecture: Three Node Types

An EMR cluster on EC2 contains up to three logical node types:

1. **Primary node** — coordinates the cluster and manages work.
2. **Core nodes** — process work and store HDFS data.
3. **Task nodes** — process work but do not store HDFS data.

The primary is the control plane inside the cluster. Core and task nodes form the worker capacity. A single-node cluster puts all roles on its only primary instance, but that layout is intended for limited workloads and has no worker-node resilience.

Node type describes responsibility; an **instance group** or **instance fleet** describes how EC2 capacity is supplied.

# 7. Primary Node: Coordinator and Control Point

The primary node coordinates the cluster rather than providing the main pool of parallel compute. Depending on installed applications, it commonly runs:

- **YARN ResourceManager** — accepts applications and allocates cluster resources
- **HDFS NameNode** — maintains filesystem namespace and block-location metadata
- **Spark driver** in YARN client mode; in cluster mode the driver runs in a YARN container
- Service endpoints and coordinators such as HiveServer2, the Hive metastore when locally configured, or Trino coordinator
- EMR agents that manage steps, report status, and coordinate configuration

Avoid unnecessary heavy processing on the primary. If it becomes overloaded, scheduling, metadata operations, and user interfaces can all slow down.

# 8. Primary Node: Capacity and Availability

Choose enough memory and CPU for coordinators, metadata, drivers, and service daemons. Driver-heavy Spark workloads may require a larger primary even when executors fit on smaller workers.

A standard cluster has one primary node. Supported high-availability configurations can use three primary nodes so key services can fail over.

**Scaling rule:** primary capacity is not the routine elasticity layer. You generally do not add or remove primary nodes after creation. Size and availability design are cluster-creation decisions. If the design is wrong, create a replacement cluster.

Keep durable state external: S3 for files, AWS Glue Data Catalog or an external metastore when shared metadata is required, and CloudWatch/S3 for durable logs.

# 9. Core Nodes: Compute Plus HDFS Storage

Core nodes perform distributed processing **and** contribute storage to HDFS. Depending on the framework, they commonly run:

- **YARN NodeManager** — launches and monitors containers
- **HDFS DataNode** — stores HDFS blocks and serves block reads/writes
- Spark executors, MapReduce tasks, Hive execution tasks, and other YARN containers
- Framework-specific worker processes when the selected application requires them

Core nodes are essential when HDFS holds intermediate or working data. CPU, memory, network bandwidth, instance-store disks, and EBS configuration all affect processing performance.

Because they hold HDFS blocks, losing or removing core nodes has a data-safety and rebalancing impact.

# 10. Core Nodes: Adding and Reducing

Core capacity can be increased after cluster creation. New nodes provide more YARN compute and more HDFS capacity.

Core capacity can also be reduced, but scale-in must be cautious because these nodes contain HDFS blocks. EMR uses decommissioning behavior to move work and reduce the risk of data loss, yet sufficient remaining HDFS capacity and replication are still required.

Before reducing core nodes:

- Confirm important outputs already exist in S3
- Check HDFS utilization, replication health, and available space
- Expect data movement and possible performance impact
- Do not shrink aggressively during storage-intensive work

For highly elastic compute, prefer adding task nodes because they carry no HDFS blocks.

# 11. Task Nodes: Elastic Compute Only

Task nodes add processing power without expanding HDFS. They commonly run:

- **YARN NodeManager**
- Spark executors, MapReduce tasks, and other YARN containers
- Application-specific workers that do not require HDFS DataNode storage

They do **not** run the HDFS DataNode role and therefore do not store HDFS blocks. This makes them the safest and most flexible capacity to add or reduce while a cluster is running.

Typical purpose: handle a temporary workload spike, accelerate a large batch, or use lower-cost Spot capacity. Interruption can still kill running tasks, but schedulers can retry them elsewhere. Keep enough stable capacity for workload continuity.

# 12. What Runs Where

| Component or role | Primary | Core | Task |
|---|:---:|:---:|:---:|
| YARN ResourceManager | ✓ | — | — |
| HDFS NameNode | ✓ | — | — |
| Service coordinator / gateway | ✓ | — | — |
| YARN NodeManager | In a single-node layout | ✓ | ✓ |
| HDFS DataNode | In a single-node layout | ✓ | — |
| Spark executors / YARN tasks | Limited or single-node use | ✓ | ✓ |
| Routine elastic scale-out | No | Possible, with HDFS care | Best choice |
| Routine scale-in | No | Possible, with HDFS care | Best choice |

Exact daemon placement varies by EMR release, installed application, deployment mode, and high-availability configuration. Always confirm the architecture of the chosen framework.

# 13. Why EMR Clusters Are Ephemeral

An ephemeral cluster is intentionally disposable: create it for work, process data, persist results, and terminate it. This pattern exists because storage and compute are separated.

Benefits include:

- Pay for compute only while it is needed
- Choose a fresh size and instance mix for each workload
- Reduce configuration drift through repeatable provisioning
- Isolate workloads and application versions

When a cluster or instance terminates, its instance-store data and EMR-attached EBS volumes are not durable. HDFS data on those resources is therefore temporary.

**Operational rule:** assume the cluster can disappear. Store code, inputs, outputs, metadata, logs, and recovery checkpoints outside the cluster.

# 14. Auto-Termination and How to Prevent It

EMR supports three common lifetime patterns: terminate after submitted steps finish, terminate after an idle timeout, or remain running until deliberately terminated. An idle auto-termination policy checks cluster activity and shuts down the cluster after the configured idle period.

To prevent unwanted idle auto-termination:

- Remove the policy: `aws emr remove-auto-termination-policy --cluster-id j-...`
- Or increase/update its timeout with `put-auto-termination-policy`
- For supported releases running non-YARN work, update `/emr/metricscollector/isbusy` while genuine work is active
- Verify networking lets the metrics collector report activity

**Do not confuse controls:** termination protection guards against accidental termination actions, but it is not a data backup strategy and does not replace correct auto-termination configuration. Use protection for important long-running clusters, and still persist all valuable data externally.

# 15. Design Checklist and Takeaways

Before launching an EMR cluster, answer these questions:

1. Which EMR release and applications are required?
2. Are durable inputs, outputs, logs, and metadata externalized to S3 or another durable service?
3. Is the primary sized for coordinators and drivers?
4. How much stable core capacity is required for HDFS and baseline compute?
5. Can burst capacity use task nodes and Spot Instances?
6. Should the cluster end after steps, after an idle timeout, or only manually?
7. Are IAM, encryption, networking, monitoring, and cost controls ready?

**Final model:** S3 is the durable data layer; the primary coordinates; core nodes compute and store HDFS blocks; task nodes provide elastic compute; EMR clusters should be treated as replaceable infrastructure.

References: [EMR architecture](https://docs.aws.amazon.com/emr/latest/ManagementGuide/emr-overview-arch.html), [node configuration](https://docs.aws.amazon.com/emr/latest/ManagementGuide/emr-plan-instances.html), [auto-termination](https://docs.aws.amazon.com/emr/latest/ManagementGuide/emr-auto-termination-policy.html), and [termination protection](https://docs.aws.amazon.com/emr/latest/ManagementGuide/UsingEMR_TerminationProtection.html).